# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EgeGln365/FlyRank_AI_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")


if HF_TOKEN is None:
    raise ValueError("HF_TOKEN bulunamadı. .env dosyanı kontrol et.")


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [4]:
march_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS contents,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES["fact_daily"]}
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-31'
""").df()

march_check

,rows,clients,contents,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [5]:
schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM {TABLES["fact_daily"]}
""").df()

numeric_columns = schema[
    schema["column_type"].str.contains(
        "BIGINT|INTEGER|DOUBLE|FLOAT|DECIMAL",
        case=False,
        regex=True
    )
]["column_name"].tolist()

print("Sayısal sütun sayısı:", len(numeric_columns))

for col in numeric_columns:
    print(col)

Sayısal sütun sayısı: 23
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events


In [6]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)




In [7]:
daily_head = con.sql(f"""
    SELECT *
    FROM {TABLES["fact_daily"]}
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-31'
    LIMIT 5
""").df()

daily_head

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,3.350000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,0.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,4.928000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,4.000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,2.272727,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [8]:
queries = []

for col in numeric_columns:
    queries.append(f"""
        SELECT
            '{col}' AS feature,
            MIN({col}) AS min_value,
            MAX({col}) AS max_value,
            AVG({col}) AS mean_value,
            MEDIAN({col}) AS median_value,
            COUNT({col}) AS non_null_count,
            COUNT(*) - COUNT({col}) AS null_count,
            SUM(CASE WHEN {col} = 0 THEN 1 ELSE 0 END) AS zero_count
        FROM {TABLES["fact_daily"]}
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-31'
    """)

profile_query = "\nUNION ALL\n".join(queries)

numeric_profile = con.sql(profile_query).df()

numeric_profile

,feature,min_value,max_value,mean_value,median_value,non_null_count,null_count,zero_count
0,gsc_impressions,0.0,40084.0,28.518119,0.0,9841378,0,6230317.0
1,gsc_clicks,0.0,274.0,0.083508,0.0,9841378,0,9423397.0
2,gsc_sum_position,0.0,481946.0,329.874230,0.0,9841378,0,6393506.0
3,gsc_avg_position,0.0,498.0,15.826651,7.5,3611061,6230317,163189.0
4,ga4_pageviews,0.0,875.0,0.217637,0.0,6822637,3018741,6409320.0
5,ga4_sessions,0.0,792.0,0.190514,0.0,6822637,3018741,6412302.0
6,ga4_users,0.0,740.0,0.186090,0.0,6822637,3018741,6412302.0
7,ga4_engaged_sessions,0.0,21.0,0.004331,0.0,6822637,3018741,6796039.0
8,ga4_total_engagement_sec,0.0,7083.0,0.698905,0.0,6822637,3018741,6716034.0
9,sessions_organic,0.0,405.0,0.085804,0.0,6822637,3018741,6609994.0


In [9]:
march_content = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- =================
    -- DATA VISIBILITY
    -- =================

    COUNT(DISTINCT report_date) AS observed_days,

    COUNT(DISTINCT report_date) FILTER(
    WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days,

    COUNT(DISTINCT report_date) FILTER(
    WHERE ga4_data_available IS TRUE
    ) AS ga4_available_days,

    -- ===============
    -- GSC FEATURES
    -- ===============

    SUM(gsc_impressions) FILTER(
    WHERE gsc_data_available IS TRUE
    ) AS impressions_march,

    SUM(gsc_clicks) FILTER(
    WHERE gsc_data_available IS TRUE
    ) AS clicks_march,

    CASE
        WHEN SUM(gsc_impressions) FILTER(
            WHERE gsc_data_available IS TRUE
            ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER(
            WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS ctr_march,

    CASE
        WHEN SUM(gsc_impressions) FILTER(
        WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER(
                WHERE gsc_data_available IS TRUE
                ) * 1.0
            /
            SUM(gsc_impressions) FILTER(
            WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS avg_position_march,

    -- ==============
    -- GA4 FEATURE
    -- ==============

    SUM(ga4_pageviews) FILTER (
    WHERE ga4_data_available IS TRUE
    ) AS pageviews_march,

    SUM(ga4_sessions) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS sessions_march,

    SUM(ga4_engaged_sessions) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS engaged_sessions_march,

    SUM(ga4_total_engagement_sec) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS engagement_sec_march,


    -- =========================
    -- GA4 FEATURES
    -- =========================

    SUM(ga4_pageviews) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS pageviews_march,

    SUM(ga4_sessions) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS sessions_march,

    SUM(ga4_engaged_sessions) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS engaged_sessions_march,

    SUM(ga4_total_engagement_sec) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS engagement_sec_march,

    -- =========================
    -- ENGAGEMENT
    -- =========================
    SUM(scroll_events) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS scroll_events_march

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2026-03-01'
                      AND DATE '2026-03-31' 

GROUP BY
    client_hash_id,
    content_hash_id



""").df()
print("Shape:", march_content.shape)

march_content.head()

Shape: (331437, 18)


,client_hash_id,content_hash_id,observed_days,gsc_available_days,ga4_available_days,impressions_march,clicks_march,ctr_march,avg_position_march,pageviews_march,sessions_march,engaged_sessions_march,engagement_sec_march,pageviews_march_1,sessions_march_1,engaged_sessions_march_1,engagement_sec_march_1,scroll_events_march
0,client_3ffa76342f366962,content_d4c9f2b414179144,31,3,0,4.0,0.0,0.0,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,client_3ffa76342f366962,content_c7f7f2880bbf0708,29,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,client_3ffa76342f366962,content_c1110d472e9fb322,29,0,1,NaN,NaN,NaN,NaN,1.0,1.0,0.0,7.0,1.0,1.0,0.0,7.0,1.0
3,client_3ffa76342f366962,content_7b6828f14547005b,29,2,0,2.0,0.0,0.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,client_3ffa76342f366962,content_6cea23fd5503f4b5,29,1,0,2.0,0.0,0.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print("Shape:", march_content.shape)

print(
    "Unique client-content pairs:",
    march_content[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate client-content pairs:",
    march_content.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

Shape: (331437, 18)
Unique client-content pairs: 331437
Duplicate client-content pairs: 0


In [11]:
march_content[
    [
        "observed_days",
        "gsc_available_days",
        "ga4_available_days",
        "impressions_march",
        "clicks_march",
        "ctr_march",
        "avg_position_march",
        "pageviews_march",
        "sessions_march",
        "engaged_sessions_march",
        "engagement_sec_march",
        "scroll_events_march"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
observed_days,331437.0,29.693058,4.738734,1.0,31.000000,31.000000,31.000000,31.0
gsc_available_days,331437.0,10.895166,13.197774,0.0,0.000000,1.000000,28.000000,31.0
ga4_available_days,331437.0,1.249004,3.601162,0.0,0.000000,0.000000,1.000000,31.0
impressions_march,176738.0,1587.986675,5431.337724,1.0,20.000000,173.000000,1039.000000,617124.0
clicks_march,176738.0,4.650002,26.722649,0.0,0.000000,0.000000,2.000000,5668.0
ctr_march,176738.0,0.004594,0.037760,0.0,0.000000,0.000000,0.002158,1.0
avg_position_march,176738.0,15.992270,18.097575,0.0,4.917879,8.177966,20.254025,309.0
pageviews_march,90489.0,16.409243,52.366172,0.0,1.000000,3.000000,10.000000,2879.0
sessions_march,90489.0,14.364265,47.007411,0.0,1.000000,2.000000,8.000000,2730.0
engaged_sessions_march,90489.0,0.326570,1.660332,0.0,0.000000,0.000000,0.000000,224.0


In [12]:
coverage_rows = []

for days in [1, 7, 14, 20, 28, 31]:

    n = (march_content["gsc_available_days"] >= days).sum()
    pct = n / len(march_content) * 100

    coverage_rows.append({
        "min_gsc_days": days,
        "n_contents": n,
        "pct_contents": pct
    })

coverage_table = pd.DataFrame(coverage_rows)

coverage_table

,min_gsc_days,n_contents,pct_contents
0,1,176738,53.324765
1,7,138815,41.882771
2,14,121844,36.762341
3,20,106546,32.146682
4,28,83842,25.296512
5,31,61796,18.644871


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
signal1_data = march_content[
    (march_content["gsc_available_days"] >= 20) &
    (march_content["impressions_march"] > 0) &
    (march_content["avg_position_march"] > 0) &
    (march_content["ctr_march"].notna())
].copy()

print("Signal 1 usable contents:", len(signal1_data))

Signal 1 usable contents: 106546


In [14]:
signal1_data["position_bucket"] = pd.cut(
    signal1_data["avg_position_march"],
    bins=[0,3,10,20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True

)

In [15]:
signal1_table = (
    signal1_data
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr_march", "median"),
        mean_ctr=("ctr_march", "mean"),
        median_impressions=("impressions_march", "median")
    )
    .reset_index()
)

signal1_table

,position_bucket,n,median_ctr,mean_ctr,median_impressions
0,1-3,9701,0.001945,0.003277,1591.0
1,4-10,48486,0.001675,0.003176,930.0
2,11-20,20839,0.000566,0.002376,484.0
3,21+,27520,0.000000,0.001251,291.0


Verdict: CONFIRMED. Median and mean CTR both decline as average search position worsens. This supports using CTR relative to search position as a baseline opportunity signal.

In [16]:
signal2_data = march_content[
    (march_content["gsc_available_days"] >= 20) &
    (march_content["impressions_march"] > 0)
].copy()

print("Signal 2 usable contents:", len(signal2_data))

Signal 2 usable contents: 106546


In [17]:
signal2_data["impression_bucket"] = pd.qcut(
    signal2_data["impressions_march"],
    q=4,
    labels=["Low","Medium","High","Very High"],
    duplicates="drop"
)

In [18]:
signal2_data.groupby(
    "impression_bucket",
    observed=True
).agg(
    n=("content_hash_id", "size"),
    min_impressions=("impressions_march", "min"),
    median_impressions=("impressions_march", "median"),
    max_impressions=("impressions_march", "max")
).reset_index()

,impression_bucket,n,min_impressions,median_impressions,max_impressions
0,Low,26669,22.0,103.0,199.0
1,Medium,26615,200.0,361.0,648.0
2,High,26627,649.0,1187.0,2296.0
3,Very High,26635,2297.0,4991.0,617124.0


In [19]:
signal2_table = (
    signal2_data
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr_march", "median"),
        mean_ctr=("ctr_march", "mean"),
        median_position=("avg_position_march", "median"),
        median_impressions=("impressions_march", "median")
    )
    .reset_index()
)

signal2_table

,impression_bucket,n,median_ctr,mean_ctr,median_position,median_impressions
0,Low,26669,0.000000,0.002393,15.165414,103.0
1,Medium,26615,0.000000,0.002124,10.358862,361.0
2,High,26627,0.001474,0.002611,7.484895,1187.0
3,Very High,26635,0.002087,0.002997,6.108398,4991.0


Verdict: MIXED. Higher-impression content generally has better search positions and higher median CTR, but the relationship is not independent of position. Impressions are therefore more useful as an opportunity-magnitude signal than as a standalone quality signal.

### Baseline Rule

Prioritize content that already has meaningful search visibility and a good search position but receives unusually low CTR for that position.

The intuition is that these pages are already being shown to users and rank reasonably well, so improving their search click-through rate may produce additional clicks without first requiring a major ranking improvement.

### Reason Code

- `LOW_CTR_HIGH_VISIBILITY`: The content has a relatively good search position and meaningful impressions, but its CTR is low relative to comparable content.

In [20]:
good_position = signal1_data[
    signal1_data["avg_position_march"] <= 10
].copy()

ctr_thresholds = good_position["ctr_march"].quantile(
    [0.10, 0.25, 0.50, 0.75]
)

print("Contents with position <= 10:", len(good_position))
print()
print(ctr_thresholds)

Contents with position <= 10: 58187

0.10    0.000000
0.25    0.000000
0.50    0.001721
0.75    0.004251
Name: ctr_march, dtype: float64


In [21]:
ctr_by_position = (
    good_position[
        good_position["ctr_march"] > 0
    ]
    .groupby("position_bucket", observed=True)["ctr_march"]
    .quantile([0.10, 0.25, 0.50])
    .unstack()
)

ctr_by_position.columns = ["p10_ctr", "p25_ctr", "median_ctr"]

ctr_by_position

,p10_ctr,p25_ctr,median_ctr
position_bucket,,,
1-3,0.000834,0.001553,0.002947
4-10,0.000876,0.001582,0.003020


In [22]:
zero_ctr_good_position = good_position[
    good_position["ctr_march"] == 0
].copy()

print("Good-position contents:", len(good_position))
print("Zero-CTR contents:", len(zero_ctr_good_position))
print(
    "Zero-CTR percentage:",
    round(len(zero_ctr_good_position) / len(good_position) * 100, 2),
    "%"
)

zero_ctr_good_position["impressions_march"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90]
)

Good-position contents: 58187
Zero-CTR contents: 17827
Zero-CTR percentage: 30.64 %


count    17827.000000
mean       483.262916
std        931.991235
min         24.000000
25%        111.000000
50%        245.000000
75%        547.000000
90%       1083.000000
max      44707.000000
Name: impressions_march, dtype: float64

In [23]:
threshold_rows = []

for imp_threshold in [200, 500, 1000, 2000]:
    n = (
        (good_position["ctr_march"] == 0) &
        (good_position["impressions_march"] >= imp_threshold)
    ).sum()

    threshold_rows.append({
        "min_impressions": imp_threshold,
        "n_candidates": n,
        "pct_of_good_position": n / len(good_position) * 100
    })

threshold_table = pd.DataFrame(threshold_rows)
threshold_table

,min_impressions,n_candidates,pct_of_good_position
0,200,10177,17.490161
1,500,4901,8.422844
2,1000,2046,3.516249
3,2000,582,1.000223


In [24]:
p25_thresholds = ctr_by_position["p25_ctr"].to_dict()

p25_thresholds

{'1-3': 0.0015527950310559005, '4-10': 0.0015822784810126582}

In [25]:
candidate_check = good_position.copy()

candidate_check["ctr_threshold"] = (
    candidate_check["position_bucket"]
    .map(p25_thresholds)
    .astype(float)
)

candidate_check["is_ctr_candidate"] = (
    (candidate_check["ctr_march"] <= candidate_check["ctr_threshold"]) &
    (candidate_check["impressions_march"] >= 500)
)

print("Total good-position contents:", len(candidate_check))
print(
    "CTR_FIX candidates:",
    candidate_check["is_ctr_candidate"].sum()
)

print(
    "Candidate percentage:",
    round(candidate_check["is_ctr_candidate"].mean() * 100, 2),
    "%"
)

Total good-position contents: 58187
CTR_FIX candidates: 14997
Candidate percentage: 25.77 %


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [29]:
ctr_candidates = candidate_check[
    candidate_check["is_ctr_candidate"]
].copy()

ctr_candidates["ctr_gap"] = (
    ctr_candidates["ctr_threshold"] -
    ctr_candidates["ctr_march"]
)

ctr_candidates[
    [
        "avg_position_march",
        "impressions_march",
        "ctr_march",
        "ctr_threshold",
        "ctr_gap"
    ]
].head(10)

,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap
113,4.009823,2036.0,0.000491,0.001582,0.001091
115,4.482574,1119.0,0.000894,0.001582,0.000689
116,2.814249,7790.0,0.001155,0.001553,0.000397
122,9.341509,530.0,0.000000,0.001582,0.001582
124,4.269311,1916.0,0.001044,0.001582,0.000538
126,3.067485,815.0,0.001227,0.001582,0.000355
130,4.073171,1353.0,0.001478,0.001582,0.000104
137,9.443925,642.0,0.000000,0.001582,0.001582
139,4.276280,1484.0,0.000000,0.001582,0.001582
142,1.816343,2472.0,0.000405,0.001553,0.001148


In [30]:
ctr_candidates["action_score"] = (
    ctr_candidates["ctr_gap"] *
    ctr_candidates["impressions_march"]
)

In [31]:
ctr_candidates[
    [
        "avg_position_march",
        "impressions_march",
        "ctr_march",
        "ctr_threshold",
        "ctr_gap",
        "action_score"
    ]
].sort_values(
    "action_score",
    ascending=False
).head(10)

,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap,action_score
277494,0.665877,212404.0,0.000113,0.001553,0.001440,305.819876
35945,2.693038,134984.0,0.000007,0.001553,0.001545,208.602484
228148,0.308426,124075.0,0.000008,0.001553,0.001545,191.663043
122986,3.166132,143019.0,0.000301,0.001582,0.001282,183.295886
199095,9.735658,107584.0,0.000139,0.001582,0.001443,155.227848
179756,7.831807,89332.0,0.000045,0.001582,0.001538,137.348101
302173,0.116003,83834.0,0.000012,0.001553,0.001541,129.177019
48159,5.948459,132593.0,0.000626,0.001582,0.000956,126.799051
14423,7.208276,83788.0,0.000072,0.001582,0.001511,126.575949
119215,8.005184,82376.0,0.000134,0.001582,0.001449,119.341772


In [34]:
ctr_candidates = ctr_candidates.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

ctr_candidates["rank"] = range(1, len(ctr_candidates) + 1)

ctr_candidates["action"] = "CTR_FIX"
ctr_candidates["reason_code"] = "LOW_CTR_HIGH_VISIBILITY"

In [35]:
ctr_candidates[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "avg_position_march",
        "impressions_march",
        "ctr_march",
        "ctr_threshold",
        "ctr_gap"
    ]
].head(20)

,rank,client_hash_id,content_hash_id,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap
0,1,client_23a62021009f63c4,content_44f34c0a90047651,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,0.001553,0.001440
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,0.001553,0.001545
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,0.001553,0.001545
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,0.001582,0.001282
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,0.001582,0.001443
5,6,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,137.348101,7.831807,89332.0,0.000045,0.001582,0.001538
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,129.177019,0.116003,83834.0,0.000012,0.001553,0.001541
7,8,client_62f4a7e64f5e0096,content_7c6373141eae744a,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.799051,5.948459,132593.0,0.000626,0.001582,0.000956
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.575949,7.208276,83788.0,0.000072,0.001582,0.001511
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,119.341772,8.005184,82376.0,0.000134,0.001582,0.001449


In [39]:
from pathlib import Path

# Final CSV'de tutacağımız sütunlar
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "action_score",
    "avg_position_march",
    "impressions_march",
    "ctr_march",
    "ctr_threshold",
    "ctr_gap"
]

# Final ranked queue
baseline_queue = ctr_candidates[output_cols].copy()

# outputs klasörü yoksa oluştur
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# CSV'nin kaydedileceği yer
output_path = output_dir / "baseline_action_score.csv"

# CSV'yi kaydet
baseline_queue.to_csv(
    output_path,
    index=False
)

print("Saved rows:", len(baseline_queue))
print("Saved to:", output_path)

Saved rows: 14997
Saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [40]:
top20_review = baseline_queue.head(20).copy()

top20_review[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "avg_position_march",
        "impressions_march",
        "ctr_march",
        "ctr_threshold",
        "ctr_gap"
    ]
]

,rank,client_hash_id,content_hash_id,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap
0,1,client_23a62021009f63c4,content_44f34c0a90047651,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,0.001553,0.001440
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,0.001553,0.001545
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,0.001553,0.001545
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,0.001582,0.001282
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,0.001582,0.001443
5,6,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,137.348101,7.831807,89332.0,0.000045,0.001582,0.001538
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,129.177019,0.116003,83834.0,0.000012,0.001553,0.001541
7,8,client_62f4a7e64f5e0096,content_7c6373141eae744a,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.799051,5.948459,132593.0,0.000626,0.001582,0.000956
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.575949,7.208276,83788.0,0.000072,0.001582,0.001511
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,119.341772,8.005184,82376.0,0.000134,0.001582,0.001449


In [41]:
top20_review["confidence_note"] = "High"

top20_review.loc[
    top20_review["avg_position_march"] < 1,
    "confidence_note"
] = "Needs review: average position is below 1"

In [42]:
top20_review["what_would_make_it_wrong"] = (
    "Low CTR may be caused by SERP/search-intent factors rather than "
    "a fixable title/snippet issue."
)

top20_review.loc[
    top20_review["avg_position_march"] < 1,
    "what_would_make_it_wrong"
] = (
    "The pick may be unreliable if the sub-1 average position is caused "
    "by a data-quality or aggregation issue."
)

In [43]:
top20_review[
    [
        "rank",
        "action",
        "reason_code",
        "action_score",
        "avg_position_march",
        "impressions_march",
        "ctr_march",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,rank,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,confidence_note,what_would_make_it_wrong
0,1,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,Needs review: average position is below 1,The pick may be unreliable if the sub-1 averag...
1,2,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,High,Low CTR may be caused by SERP/search-intent fa...
2,3,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,Needs review: average position is below 1,The pick may be unreliable if the sub-1 averag...
3,4,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,High,Low CTR may be caused by SERP/search-intent fa...
4,5,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,High,Low CTR may be caused by SERP/search-intent fa...
5,6,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,137.348101,7.831807,89332.0,0.000045,High,Low CTR may be caused by SERP/search-intent fa...
6,7,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,129.177019,0.116003,83834.0,0.000012,Needs review: average position is below 1,The pick may be unreliable if the sub-1 averag...
7,8,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.799051,5.948459,132593.0,0.000626,High,Low CTR may be caused by SERP/search-intent fa...
8,9,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.575949,7.208276,83788.0,0.000072,High,Low CTR may be caused by SERP/search-intent fa...
9,10,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,119.341772,8.005184,82376.0,0.000134,High,Low CTR may be caused by SERP/search-intent fa...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [46]:
weak_picks = top20_review[
    top20_review["avg_position_march"] < 1
].copy()

weak_picks[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "avg_position_march",
        "impressions_march",
        "ctr_march"
    ]
]

,rank,client_hash_id,content_hash_id,action_score,avg_position_march,impressions_march,ctr_march
0,1,client_23a62021009f63c4,content_44f34c0a90047651,305.819876,0.665877,212404.0,0.000113
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,191.663043,0.308426,124075.0,0.000008
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,129.177019,0.116003,83834.0,0.000012


In [47]:
weak_ids = weak_picks[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()

weak_daily = con.sql(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.report_date,
        d.gsc_impressions,
        d.gsc_sum_position,
        d.gsc_avg_position,
        d.gsc_data_available
    FROM {TABLES["fact_daily"]} d
    INNER JOIN weak_ids w
        ON d.client_hash_id = w.client_hash_id
       AND d.content_hash_id = w.content_hash_id
    WHERE d.report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-31'
    ORDER BY
        d.client_hash_id,
        d.content_hash_id,
        d.report_date
""").df()

In [48]:
weak_position_check = (
    weak_daily
    .groupby(["client_hash_id", "content_hash_id"])
    .agg(
        days=("report_date", "nunique"),
        impressions=("gsc_impressions", "sum"),
        sum_position=("gsc_sum_position", "sum"),
        min_daily_position=("gsc_avg_position", "min"),
        median_daily_position=("gsc_avg_position", "median"),
        max_daily_position=("gsc_avg_position", "max")
    )
    .reset_index()
)

weak_position_check

,client_hash_id,content_hash_id,days,impressions,sum_position,min_daily_position,median_daily_position,max_daily_position
0,client_23a62021009f63c4,content_44f34c0a90047651,31,212404,141435,0.083350,8.190751,18.497537
1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,31,83834,9725,0.000000,5.833333,49.200000
2,client_73cda7b4e4f265ea,content_fec55986a1868d62,31,124075,38268,0.013056,6.000000,32.500000


Ranks 1, 3, and 7 are weak picks because their aggregated March average position is below 1. Inspection of the daily records shows that sub-1 position values already occur in the source data, while median daily positions are substantially higher. The monthly calculation itself is consistent with SUM(gsc_sum_position) / SUM(gsc_impressions), but the relationship between the source position fields requires further validation. I would not automatically trust these picks without checking the position-field semantics.

In [49]:
leakage_audit = {
    "feature_window": "2026-03-01 to 2026-03-31",
    "uses_future_query_90d": False,
    "uses_future_outcome_or_label": False,
    "uses_product_flag_as_feature": False,
    "uses_ids_as_features": False,
    "score_inputs": [
        "avg_position_march",
        "ctr_march",
        "impressions_march",
        "ctr_threshold",
        "ctr_gap"
    ]
}

for key, value in leakage_audit.items():
    print(f"{key}: {value}")

feature_window: 2026-03-01 to 2026-03-31
uses_future_query_90d: False
uses_future_outcome_or_label: False
uses_product_flag_as_feature: False
uses_ids_as_features: False
score_inputs: ['avg_position_march', 'ctr_march', 'impressions_march', 'ctr_threshold', 'ctr_gap']


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.